### **The Runnable and RunnableSequence Classes**

In [1]:
%load_ext dotenv
%dotenv

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [3]:
chat_template = ChatPromptTemplate.from_messages([('human', 
                                                    "I've recently adopted a {pet} which is a {breed}. Could you suggest several training tips?")])

In [4]:
chat = ChatOpenAI(
    model = 'gpt-4o',
    seed = 7,
    temperature = 0,
    max_completion_tokens = 100
)

In [5]:
chain = chat_template | chat

In [6]:
type(chain)

langchain_core.runnables.base.RunnableSequence

#### In **RunnableSequence** the output of each is the input of next

#### Prompt Template, Chat Model, Output Parser are the instances of **Runnable** 

In [7]:
type(chat_template)

langchain_core.prompts.chat.ChatPromptTemplate

#### **ChatPromptTemplate** : **BaseChatPromptTemplate** : **BasePromptTemplate** : **RunnableSerializable** : **Runnable**

## Inheritance

In [8]:
type(chat)

langchain_openai.chat_models.base.ChatOpenAI

### **Piping Chains and the RunnablePassthrough Class**

In [9]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [10]:
RunnablePassthrough().invoke('hi')

'hi'

In [11]:
RunnablePassthrough().invoke([1, 2, 3])

[1, 2, 3]

#### That's the LangChain identity class

In [12]:
chat_template_tools = ChatPromptTemplate.from_template('''
    What are the five most important tools a {job_title} needs?
    Answer only by listing the tools.
''')

chat_template_strategy = ChatPromptTemplate.from_template('''
    Considering the tools provided, develop a strategy for effectively learning and mastering them:
    {tools}
''')

In [13]:
chat_template_tools

ChatPromptTemplate(input_variables=['job_title'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['job_title'], input_types={}, partial_variables={}, template='\n    What are the five most important tools a {job_title} needs?\n    Answer only by listing the tools.\n'), additional_kwargs={})])

In [14]:
chat = ChatOpenAI(
    model='gpt-4o',
    seed=7,
    temperature=0,
    max_completion_tokens=100
)

In [15]:
string_parser = StrOutputParser()

In [16]:
chain_tools = chat_template_tools | chat | string_parser
chain_strategy = chat_template_strategy | chat | string_parser

In [17]:
print(chain_tools.invoke({'job_title': 'data scientist'}))

1. Python
2. R
3. Jupyter Notebook
4. SQL
5. TensorFlow


In [18]:
chain_strategy.invoke({'tools':'''
    1. Python
    2. R
    3. Jupyter Notebook
    4. SQL
    5. TensorFlow
'''})

'Developing a strategy to effectively learn and master these tools involves a structured approach that combines theoretical understanding, practical application, and continuous improvement. Here\'s a step-by-step strategy for each tool:\n\n### 1. Python\n\n**Phase 1: Basics**\n- **Resources**: Start with online courses like Codecademy, Coursera, or edX to learn Python basics.\n- **Books**: "Automate the Boring Stuff with Python" by Al Sweigart.\n- **Practice**'

In [19]:
chain_strategy.invoke({'tools':chain_tools.invoke({'job_title': 'data scientist'})})

'Developing a strategy to effectively learn and master tools like Python, R, SQL, Jupyter Notebook, and TensorFlow involves a structured approach that combines theoretical understanding with practical application. Here’s a step-by-step strategy:\n\n### 1. Set Clear Goals\n- **Define Objectives**: Determine why you want to learn these tools. Is it for data analysis, machine learning, data science, or another purpose?\n- **Prioritize**: Decide which tool is most relevant to your goals and start'

In [20]:
chain_tools = chat_template_tools | chat | string_parser | {'tools': RunnablePassthrough()}
chain_strategy = chat_template_strategy | chat | string_parser

In [21]:
print(chain_tools.invoke({'job_title': 'data scientist'}))

{'tools': '1. Python\n2. R\n3. SQL\n4. Jupyter Notebook\n5. TensorFlow'}


In [22]:
chain_combined = chain_tools | chain_strategy

In [23]:
print(chain_combined.invoke({'job_title': 'data scientist'}))

Developing a strategy to effectively learn and master tools like Python, R, SQL, Jupyter Notebook, and TensorFlow involves a structured approach that combines theoretical understanding with practical application. Here’s a step-by-step strategy to guide you through the process:

### 1. **Python**

- **Foundation Building:**
  - Start with the basics of Python programming. Use resources like "Automate the Boring Stuff with Python" or online platforms like Codecademy or Coursera.
  - Focus


In [24]:
chain_long = (chat_template_tools | chat | string_parser | {'tools': RunnablePassthrough()} | 
              chat_template_strategy | chat | string_parser)

In [25]:
chain_long

ChatPromptTemplate(input_variables=['job_title'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['job_title'], input_types={}, partial_variables={}, template='\n    What are the five most important tools a {job_title} needs?\n    Answer only by listing the tools.\n'), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.7.7', 'langchain': '0.2.8', 'langchain-openai': '1.7.4'}}, profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool

In [26]:
chain_long.invoke({'job_title': 'full stack programmer'})

"Developing a strategy to effectively learn and master these tools involves a structured approach that combines theoretical understanding, practical application, and continuous improvement. Here's a step-by-step strategy for each tool:\n\n### 1. Integrated Development Environment (IDE) or Code Editor\n\n**Objective:** Become proficient in using an IDE or code editor to enhance productivity and code quality.\n\n- **Research and Selection:** Choose an IDE or code editor that suits your needs (e.g., Visual Studio Code, IntelliJ IDEA, Sublime"

### **Visualizing chains**

In [27]:
chain_long.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
     +-------------+       
     | Passthrough |       
     +-------------+       
            *              
            *              
            *       

### **RunnableParallel**

In [28]:
from langchain_core.runnables import RunnableParallel

In [31]:
chat_template_books = ChatPromptTemplate.from_template(
    '''
    Suggest three of best intermediate-level {programming language} books.
    Answer only by listing the books.
    '''
)

chat_template_projects = ChatPromptTemplate.from_template(
    '''
    Suggest three interesting {programming language} projects for intermediate-level programmers.
    Answer only by listing the projects.
    '''
)

In [32]:
chain_books = chat_template_books | chat | string_parser

chain_projects = chat_template_projects | chat | string_parser

In [33]:
chain_parallel = RunnableParallel({'books':chain_books, 'projects': chain_projects})

In [34]:
chain_projects.invoke({'programming language': 'Python'})

'1. Personal Finance Tracker\n2. Web Scraper with Data Visualization\n3. Chatbot using Natural Language Processing'

In [36]:
chain_parallel.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            

In [37]:
%%time
chain_books.invoke({'programming language': 'Python'})

CPU times: total: 31.2 ms
Wall time: 1.04 s


'1. "Fluent Python" by Luciano Ramalho\n2. "Effective Python: 90 Specific Ways to Write Better Python" by Brett Slatkin\n3. "Python Tricks: A Buffet of Awesome Python Features" by Dan Bader'

In [38]:
%%time
chain_projects.invoke({'programming language': 'Python'})

CPU times: total: 0 ns
Wall time: 917 ms


'1. Personal Finance Tracker\n2. Web Scraper with Data Visualization\n3. Chatbot using Natural Language Processing (NLP)'

In [39]:
%%time
chain_parallel.invoke({'programming language': 'Python'})

CPU times: total: 62.5 ms
Wall time: 1.04 s


{'books': '1. "Fluent Python" by Luciano Ramalho\n2. "Effective Python: 90 Specific Ways to Write Better Python" by Brett Slatkin\n3. "Python Cookbook" by David Beazley and Brian K. Jones',
 'projects': '1. Personal Finance Tracker\n2. Web Scraper with Data Visualization\n3. Chatbot using Natural Language Processing'}

### **Piping a RunnableParallel with Other Runnables**

In [40]:
chat_template_time = ChatPromptTemplate.from_template('''
    I'm an intermediate level programmer.

    Consider the following literature:
    {books}

    Also, consider the following projects:
    {projects}

    Roughly how much time would it take me to complete the literature and the projects?
''')

In [41]:
chat = ChatOpenAI(
    model='gpt-4o',
    seed=7,
    temperature=0,
    max_completion_tokens=500
)

In [42]:
chain_time = (RunnableParallel({'books':chain_books, 
                                'projects': chain_projects}) 
              | chat_template_time
              | chat
              | string_parser
             )

In [43]:
print(chain_time.invoke({'programming language': 'Python'}))

Estimating the time required to complete both the literature and the projects depends on several factors, including your current skill level, the amount of time you can dedicate daily or weekly, and your familiarity with the topics covered in the books and projects. However, I can provide a rough estimate based on typical scenarios:

### Literature

1. **"Fluent Python" by Luciano Ramalho**
   - **Time Estimate**: 4-6 weeks
   - **Reason**: This book is comprehensive and covers advanced Python concepts. If you dedicate a few hours each day, you can work through it in about a month or a bit longer.

2. **"Effective Python: 90 Specific Ways to Write Better Python" by Brett Slatkin**
   - **Time Estimate**: 2-3 weeks
   - **Reason**: This book is structured in short, focused chapters, making it easier to digest. You can read and practice a few items each day.

3. **"Python Cookbook" by David Beazley and Brian K. Jones**
   - **Time Estimate**: 3-5 weeks
   - **Reason**: This book is more 

In [44]:
chain_time.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            

In [45]:
chain_time1 = (RunnableParallel({'books':chain_books, 
                                'projects': chain_projects} 
              | chat_template_time
              | chat
              | string_parser
             )

In [48]:
chain_time2 = ({'books':chain_books, 
                                'projects': chain_projects}
              | chat_template_time
              | chat
              | string_parser
             )
# We don't need to pack the parallel chains into RunnableParallel. If there is another Runnable in the chain, it will be automatically mapped

In [49]:
chain_time2.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            

### **RunnableLambda**

##### It is used for transforming ordinary functions into Runnable objects

In [50]:
find_sum = lambda x: sum(x)

In [51]:
find_sum([1, 2, 5])

8

In [53]:
find_square = lambda x: x**2

In [54]:
find_square(8)

64

In [55]:
from langchain_core.runnables import RunnableLambda

In [56]:
runnable_sum = RunnableLambda(lambda x: sum(x))

In [57]:
runnable_sum.invoke([1, 2, 5])

8

In [58]:
runnable_square = RunnableLambda(lambda x: x**2)

In [60]:
runnable_square.invoke(8)

64

In [61]:
chain = runnable_sum | runnable_square

In [62]:
chain.invoke([1, 2, 5])

64

In [63]:
chain.get_graph().print_ascii()

+-------------+  
| LambdaInput |  
+-------------+  
        *        
        *        
        *        
   +--------+    
   | Lambda |    
   +--------+    
        *        
        *        
        *        
   +--------+    
   | Lambda |    
   +--------+    
        *        
        *        
        *        
+--------------+ 
| LambdaOutput | 
+--------------+ 
